### Unit Test 01: SQL UDF Column Name Standardization

This unit test validates the SQL UDF `coffee.silver.standardize_column_name(col_name)` used for column name standardization across the pipeline.

The test executes the UDF on multiple sample input column names containing spaces, hyphens, special characters, and inconsistent formatting, and verifies that:

- Column names are converted into a consistent snake_case format
- Spaces and hyphens are standardized into underscores
- Special characters are removed
- Extra underscores are collapsed and trimmed

This ensures a consistent naming standard is enforced before downstream Silver and Gold transformations.


In [0]:
%python
# ------------------------------------------------------------
# Unit Test 01: Validate SQL UDF output
#
# UDF under test:
#   coffee.silver.standardize_column_name(col_name STRING)
#
# What this test validates:
# - Column names are standardized into snake_case
# - Spaces and hyphens are converted to underscores
# - Special characters are removed (except underscore)
# - Multiple underscores are collapsed
# - Leading/trailing underscores are trimmed
#
# Why this matters:
# This UDF is reused across the pipeline to standardize column names
# before writing to silver and gold tables. If this breaks, many
# downstream transformations can fail.
# ------------------------------------------------------------

# Test cases: (input_column_name, expected_standardized_output)
test_cases = [
    (" Transaction ID ", "transaction_id"),   # trims spaces + converts to snake_case
    ("Created At", "created_at"),             # space replaced with underscore
    ("Store-ID", "store_id"),                 # hyphen replaced with underscore
    ("Total$Amount", "totalamount"),          # special character removed
    (" user id ", "user_id"),                 # leading/trailing spaces removed
    ("City---Name", "city_name"),             # multiple hyphens collapse into one underscore
    ("__Store__ID__", "store_id")             # trims underscores + collapses multiple underscores
]

for inp, expected in test_cases:
    # Call the SQL UDF directly using spark.sql
    actual = spark.sql(
        f"SELECT coffee.silver.standardize_column_name('{inp}') AS col"
    ).collect()[0]["col"]

    # Assertion: output must match expected value
    assert actual == expected, (
        f"FAILED for input={inp}. Expected={expected}, Got={actual}"
    )

print(" Unit Test 01 passed: SQL UDF standardize_column_name works correctly.")
